In [2]:
import os
from pathlib import Path
import SimpleITK as sitk


def convert_one_patient(patient_dir: Path, output_dir: Path):
    """
    Caută în interiorul unui director de pacient un folder
    'CT obtinut in orma anonimizarii/S0001' și convertește seria DICOM în NIfTI.
    """
    # Căutăm toate directoarele numite exact 'S0001'
    candidate_series_dirs = [
        d for d in patient_dir.rglob("S0001") if d.is_dir()
    ]

    if not candidate_series_dirs:
        print(f"[WARN] Nu am găsit niciun folder 'S0001' în {patient_dir}")
        return

    # Alegem primul găsit (dacă ai multiple, aici poți pune o regulă mai specifică)
    series_dir = candidate_series_dirs[0]
    print(f"[INFO] Pacient {patient_dir.name}: folosesc seria {series_dir}")

    # Luăm toate fișierele DICOM din S0001 (presupunem extensii arbitrare)
    dicom_files = sorted(
        [str(f) for f in series_dir.glob("*") if f.is_file()]
    )

    if not dicom_files:
        print(f"[WARN] Nu există fișiere DICOM în {series_dir}")
        return

    # Citim seria ca volum 3D
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(dicom_files)

    try:
        image = reader.Execute()
    except RuntimeError as e:
        print(f"[ERROR] Eroare la citire DICOM pentru {patient_dir.name}: {e}")
        return

    # Construim calea de output: <COD_PACIENT>_CT.nii.gz
    patient_code = patient_dir.name
    output_path = output_dir / f"{patient_code}_CT.nii.gz"

    # Ne asigurăm că directorul de output există
    output_dir.mkdir(parents=True, exist_ok=True)

    sitk.WriteImage(image, str(output_path))
    print(f"[OK] Salvat: {output_path}")


def batch_convert(input_root: str, output_root: str):
    input_root = Path(input_root)
    output_root = Path(output_root)

    if not input_root.is_dir():
        raise ValueError(f"Input root nu există sau nu e director: {input_root}")

    # Iterăm prin toate subdirectoarele din input_root (câte unul per pacient)
    for patient_dir in sorted([d for d in input_root.iterdir() if d.is_dir()]):
        convert_one_patient(patient_dir, output_root)


if __name__ == "__main__":
    INPUT_ROOT = r"E:\test" 
    OUTPUT_ROOT = r"E:\output_ct"  

    batch_convert(INPUT_ROOT, OUTPUT_ROOT)


[INFO] Pacient 11111: folosesc seria E:\test\11111\CT\S0001
[OK] Salvat: E:\output_ct\11111_CT.nii.gz
[INFO] Pacient 12111: folosesc seria E:\test\12111\CT\S0001
[OK] Salvat: E:\output_ct\12111_CT.nii.gz
[INFO] Pacient 13111: folosesc seria E:\test\13111\CT\S0001
[OK] Salvat: E:\output_ct\13111_CT.nii.gz


In [3]:
pip install pydicom nibabel dicom2nifti

Defaulting to user installation because normal site-packages is not writeable
  Using cached nibabel-5.3.2-py3-none-any.whl.metadata (9.1 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------- ------------- 1.6/2.4 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 10.4 MB/s eta 0:00:00
Using cached nibabel-5.3.2-py3-none-any.whl (3.3 MB)
   ---------------------------------------- 0.0/34.2 MB ? eta -:--:--
   -- ------------------------------------- 2.4/34.2 MB 12.2 MB/s eta 0:00:03
   ----- ---------------------------------- 5.0/34.2 MB 12.1 MB/s eta 0:00:03
   -------- ------------------------------- 7.3/34.2 MB 12.2 MB/s eta 0:00:03
   ----------- ---------------------------- 10.0/34.2 MB 12.2 MB/s eta 0:00:02
   -------------- ------------------------- 12.3/34.2 MB 11.9 MB/s eta 0:00:02
   ----------------- ---------------------- 14.7/34.2 MB 11.8 MB/s eta 0:00:02
   ------------------- -------------------


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pydicom
import nibabel as nib
import dicom2nifti
import os

In [5]:
def convert_dicom_to_nifti(dicom_dir, output_file):
    dicom2nifti.convert_directory(dicom_dir, os.path.dirname(output_file), compression=True)

In [6]:
dicom_directory = r'E:\test\11111\CT\S0001'
output_file = r'E:\output_test\11111_CT.nii'

dicom2nifti.convert_directory(dicom_directory, os.path.dirname(output_file))

In [2]:
import os
from pathlib import Path
import dicom2nifti


def convert_dicom_to_nifti(dicom_dir, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    dicom2nifti.dicom_series_to_nifti(
        dicom_dir,
        output_file
    )


def find_s0001_dir(patient_dir: Path) -> Path | None:
    """
    Caută recursive în directorul pacientului un folder numit exact 'S0001'.
    Îl întoarce pe primul găsit sau None dacă nu există.
    """
    for d in patient_dir.rglob("S0001"):
        if d.is_dir():
            return d
    return None


def batch_convert(input_root: str, output_root: str):
    input_root = Path(input_root)
    output_root = Path(output_root)

    if not input_root.is_dir():
        raise ValueError(f"Input root nu există sau nu e director: {input_root}")

    for patient_dir in sorted([d for d in input_root.iterdir() if d.is_dir()]):
        patient_code = patient_dir.name
        print(f"\n[INFO] Procesez pacient: {patient_code}")

        s0001_dir = find_s0001_dir(patient_dir)
        if s0001_dir is None:
            print(f"[WARN] Nu am găsit niciun folder 'S0001' în {patient_dir}")
            continue

        print(f"[INFO] Găsit S0001 în: {s0001_dir}")

        out_file = output_root / f"{patient_code}_CT.nii.gz"

        try:
            convert_dicom_to_nifti(str(s0001_dir), str(out_file))
        except Exception as e:
            print(f"[ERROR] Eroare la conversie pentru {patient_code}: {e}")
        else:
            print(f"[OK] Salvat: {out_file}")

    


In [3]:
INPUT_ROOT = r"E:\lung project" 
OUTPUT_ROOT = r"E:\output_ct"   

batch_convert(INPUT_ROOT, OUTPUT_ROOT)


[INFO] Procesez pacient: 100111
[INFO] Găsit S0001 în: E:\lung project\100111\CT\S0001
[OK] Salvat: E:\output_ct\100111_CT.nii.gz

[INFO] Procesez pacient: 101011
[INFO] Găsit S0001 în: E:\lung project\101011\CT\S0001
[OK] Salvat: E:\output_ct\101011_CT.nii.gz

[INFO] Procesez pacient: 102111
[INFO] Găsit S0001 în: E:\lung project\102111\CT\S0001
[OK] Salvat: E:\output_ct\102111_CT.nii.gz

[INFO] Procesez pacient: 103111
[INFO] Găsit S0001 în: E:\lung project\103111\CT\S0001
[OK] Salvat: E:\output_ct\103111_CT.nii.gz

[INFO] Procesez pacient: 104011
[INFO] Găsit S0001 în: E:\lung project\104011\CT\S0001
[OK] Salvat: E:\output_ct\104011_CT.nii.gz

[INFO] Procesez pacient: 31011
[INFO] Găsit S0001 în: E:\lung project\31011\CT\S0001
[OK] Salvat: E:\output_ct\31011_CT.nii.gz

[INFO] Procesez pacient: 32111
[INFO] Găsit S0001 în: E:\lung project\32111\CT\S0001
[OK] Salvat: E:\output_ct\32111_CT.nii.gz

[INFO] Procesez pacient: 33111
[INFO] Găsit S0001 în: E:\lung project\33111\CT\S0001
[OK

In [4]:
INPUT_ROOT = r"E:\lung project NEGRU" 
OUTPUT_ROOT = r"E:\output_ct"   

batch_convert(INPUT_ROOT, OUTPUT_ROOT)


[INFO] Procesez pacient: 10111
[INFO] Găsit S0001 în: E:\lung project NEGRU\10111\CT\S0001
[OK] Salvat: E:\output_ct\10111_CT.nii.gz

[INFO] Procesez pacient: 2011
[INFO] Găsit S0001 în: E:\lung project NEGRU\2011\CT\S0001
[OK] Salvat: E:\output_ct\2011_CT.nii.gz

[INFO] Procesez pacient: 22011
[INFO] Găsit S0001 în: E:\lung project NEGRU\22011\CT\S0001
[OK] Salvat: E:\output_ct\22011_CT.nii.gz

[INFO] Procesez pacient: 23111
[INFO] Găsit S0001 în: E:\lung project NEGRU\23111\CT\S0001
[OK] Salvat: E:\output_ct\23111_CT.nii.gz

[INFO] Procesez pacient: 24111
[INFO] Găsit S0001 în: E:\lung project NEGRU\24111\CT\S0001
[OK] Salvat: E:\output_ct\24111_CT.nii.gz

[INFO] Procesez pacient: 25111
[INFO] Găsit S0001 în: E:\lung project NEGRU\25111\CT\S0001
[OK] Salvat: E:\output_ct\25111_CT.nii.gz

[INFO] Procesez pacient: 26011
[INFO] Găsit S0001 în: E:\lung project NEGRU\26011\CT\S0001
[OK] Salvat: E:\output_ct\26011_CT.nii.gz

[INFO] Procesez pacient: 27010
[INFO] Găsit S0001 în: E:\lung pr

In [5]:
INPUT_ROOT = r"E:\lung project ROSU" 
OUTPUT_ROOT = r"E:\output_ct"   

batch_convert(INPUT_ROOT, OUTPUT_ROOT)


[INFO] Procesez pacient: 11111
[INFO] Găsit S0001 în: E:\lung project ROSU\11111\CT\S0001
[OK] Salvat: E:\output_ct\11111_CT.nii.gz

[INFO] Procesez pacient: 12111
[INFO] Găsit S0001 în: E:\lung project ROSU\12111\CT\S0001
[OK] Salvat: E:\output_ct\12111_CT.nii.gz

[INFO] Procesez pacient: 13111
[INFO] Găsit S0001 în: E:\lung project ROSU\13111\CT\S0001
[OK] Salvat: E:\output_ct\13111_CT.nii.gz

[INFO] Procesez pacient: 15011
[INFO] Găsit S0001 în: E:\lung project ROSU\15011\CT\S0001
[OK] Salvat: E:\output_ct\15011_CT.nii.gz

[INFO] Procesez pacient: 16111
[INFO] Găsit S0001 în: E:\lung project ROSU\16111\CT\S0001
[OK] Salvat: E:\output_ct\16111_CT.nii.gz

[INFO] Procesez pacient: 17011
[INFO] Găsit S0001 în: E:\lung project ROSU\17011\CT\S0001
[OK] Salvat: E:\output_ct\17011_CT.nii.gz

[INFO] Procesez pacient: 19111
[INFO] Găsit S0001 în: E:\lung project ROSU\19111\CT\S0001
[OK] Salvat: E:\output_ct\19111_CT.nii.gz

[INFO] Procesez pacient: 453011
[INFO] Găsit S0001 în: E:\lung proje

Slice increment not consistent through all slices
---------------------------------------------------------
[-271.5  -271.5  -526.75] [ 0.    0.   -3.75]
[-271.5 -271.5 -373. ] [   0.      0.   -153.75]
Instance Number: 99
---------------------------------------------------------


[ERROR] Eroare la conversie pentru 97011: SLICE_INCREMENT_INCONSISTENT

[INFO] Procesez pacient: 98011
[INFO] Găsit S0001 în: E:\lung project ROSU\98011\CT\S0001
[OK] Salvat: E:\output_ct\98011_CT.nii.gz
